# Multimodal RAG Ingestion — Research Notebook

This notebook is the **research/prototyping version** of the four production files you supplied.

Unlike the previous notebook, we will **not import the four modules**. Instead, we will recreate their important functions directly in notebook cells and execute them sequentially.

The intent is:

> **Understand the input → write the function → execute it → inspect its output → pass that output into the next function.**

Production modules being reconstructed:

- `loaders.py`
- `captioner.py`
- `ingest.py`
- `ingest_multimodal.py`


## End-to-end workflow

```text
charts / tables / PDFs
          │
          ▼
   metadata JSON lookup
          │
          ▼
    Document construction
          │
          ├───────────────┐
          ▼               ▼
 needs_caption=True   needs_caption=False
          │               │
          ▼               │
    image → base64        │
          │               │
          ▼               │
      vision LLM          │
          │               │
          ▼               │
       caption            │
          │               │
          └───────┬───────┘
                  ▼
          searchable Documents
                  │
                  ▼
          content filtering
                  │
                  ▼
          text embeddings
                  │
                  ▼
               FAISS
                  │
                  ▼
          similarity_search()
                  │
                  ▼
          List[Document]
```


In [1]:
# 1. Environment and paths — same path logic as the edited notebook

from pathlib import Path
import json
import os
import sys
import base64
import logging
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

PROJECT_ROOT = Path.cwd().parents[0]
SRC_DIR = PROJECT_ROOT / "src"

if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

CORPUS_DIR = PROJECT_ROOT / "documents" / "multimodal"
INDEX_DIR = PROJECT_ROOT / "data" / "vectorstore" / "faiss_multimodal"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CORPUS_DIR  :", CORPUS_DIR)
print("INDEX_DIR   :", INDEX_DIR)
print("Corpus exists:", CORPUS_DIR.exists())


PROJECT_ROOT: /Users/azizulshaikh/Projects/agentic_bi_platform
CORPUS_DIR  : /Users/azizulshaikh/Projects/agentic_bi_platform/documents/multimodal
INDEX_DIR   : /Users/azizulshaikh/Projects/agentic_bi_platform/data/vectorstore/faiss_multimodal
Corpus exists: True


# Part I — `loaders.py`

## 1. Define the common object first

Every source eventually becomes a LangChain `Document`.

The fundamental contract is:

```python
Document(
    page_content: str,
    metadata: dict
)
```

We will inspect this object before implementing any loader.


In [2]:
# 2. Import only the external dependency — NOT our production loader module

from langchain_core.documents import Document

demo = Document(
    page_content="example text",
    metadata={"source_type": "demo"}
)

print("Object type:", type(demo).__name__)
print("page_content:", demo.page_content)
print("metadata:", demo.metadata)


Object type: Document
page_content: example text
metadata: {'source_type': 'demo'}


## 2. Metadata loading

Research question:

> How do we turn `charts_metadata.json`, `tables_metadata.json`, and `pdfs_metadata.json` into Python dictionaries?

The production behavior is intentionally forgiving: if metadata is missing, return an empty dictionary.


In [3]:
# 3. Recreate _load_metadata_json()

logger = logging.getLogger("multimodal_rag_research")
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

def load_metadata_json(meta_dir: Path, filename: str) -> dict:
    path = meta_dir / filename

    if not path.exists():
        logger.warning("Metadata file not found: %s", path)
        return {}

    with open(path, "r") as f:
        return json.load(f)

META_DIR = CORPUS_DIR / "metadata"

charts_meta = load_metadata_json(META_DIR, "charts_metadata.json")
tables_meta = load_metadata_json(META_DIR, "tables_metadata.json")
pdfs_meta = load_metadata_json(META_DIR, "pdfs_metadata.json")

print("charts_meta:", type(charts_meta).__name__, len(charts_meta))
print("tables_meta:", type(tables_meta).__name__, len(tables_meta))
print("pdfs_meta  :", type(pdfs_meta).__name__, len(pdfs_meta))


charts_meta: dict 10
tables_meta: dict 5
pdfs_meta  : dict 3


In [4]:
# 4. Inspect one metadata record

for name, obj in [
    ("charts", charts_meta),
    ("tables", tables_meta),
    ("pdfs", pdfs_meta),
]:
    if obj:
        key = next(iter(obj))
        print(f"\n{name}: key = {key}")
        print(json.dumps(obj[key], indent=2))



charts: key = 01_monthly_revenue_trend.png
{
  "title": "Monthly Revenue & Order Volume \u2014 Olist Platform (Jan 2017 \u2013 Aug 2018)",
  "chart_type": "dual-axis line chart",
  "x_axis": "Month (Jan 2017 \u2013 Aug 2018)",
  "y_axis_left": "Revenue in thousands BRL",
  "y_axis_right": "Order count in thousands",
  "key_insight": "Revenue grew consistently from R$320K in Jan 2017 to a peak of R$810K in Mar 2018, followed by a gradual decline to R$660K by Aug 2018. Order volume mirrors the revenue trend, peaking at ~9,500 orders in Mar 2018.",
  "data_source": "synthetic \u2014 modelled on Olist Brazilian E-Commerce dataset",
  "time_period": "January 2017 \u2013 August 2018"
}

tables: key = T01_kpi_summary_table.png
{
  "title": "Olist Platform \u2014 Q2 2018 KPI Snapshot",
  "table_type": "KPI dashboard table",
  "key_insight": "Q2 2018 shows revenue of R$2,250,000 (down 4.7% vs Q1), total orders 26,300 (down 5.4% vs Q1), return rate 5.2% (up 1.1 percentage points), and average d

## 3. Recreate `load_image_documents()`

For each image we will create:

```text
Document
├── page_content = key_insight (temporary placeholder)
└── metadata
    ├── source_type
    ├── file_path
    ├── file_name
    ├── title
    ├── key_insight
    ├── chart_type
    ├── time_period
    ├── data_source
    └── needs_caption=True
```

No LLM is called here.


In [5]:
# 5. Recreate load_image_documents()

SUPPORTED_IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".webp"}

def load_image_documents(
    image_dir: Path,
    source_type: str,
    metadata_lookup: dict
) -> list[Document]:

    docs = []

    if not image_dir.exists():
        logger.warning("No image directory found: %s", image_dir)
        return docs

    image_files = sorted(
        p for p in image_dir.iterdir()
        if p.suffix.lower() in SUPPORTED_IMAGE_SUFFIXES
    )

    for img_path in image_files:
        meta = metadata_lookup.get(img_path.name, {})

        doc = Document(
            page_content=meta.get("key_insight", ""),
            metadata={
                "source_type": source_type,
                "file_path": str(img_path.resolve()),
                "file_name": img_path.name,
                "title": meta.get("title", img_path.stem),
                "key_insight": meta.get("key_insight", ""),
                "chart_type": meta.get("chart_type", ""),
                "time_period": meta.get("time_period", ""),
                "data_source": meta.get("data_source", "synthetic"),
                "needs_caption": True,
            },
        )

        docs.append(doc)

    logger.info("Loaded %d %s documents", len(docs), source_type)
    return docs


In [6]:
# 6. Execute the image loader

chart_docs = load_image_documents(CORPUS_DIR / "charts", "chart", charts_meta)
table_docs = load_image_documents(CORPUS_DIR / "tables", "table", tables_meta)

print("Charts:", len(chart_docs))
print("Tables:", len(table_docs))

if chart_docs:
    d = chart_docs[0]
    print("\nExample chart Document")
    print("page_content type:", type(d.page_content).__name__)
    print("page_content:", d.page_content)
    print("metadata:")
    for k, v in d.metadata.items():
        print(f"  {k}: {v!r}")


INFO: Loaded 10 chart documents
INFO: Loaded 5 table documents


Charts: 10
Tables: 5

Example chart Document
page_content type: str
page_content: Revenue grew consistently from R$320K in Jan 2017 to a peak of R$810K in Mar 2018, followed by a gradual decline to R$660K by Aug 2018. Order volume mirrors the revenue trend, peaking at ~9,500 orders in Mar 2018.
metadata:
  source_type: 'chart'
  file_path: '/Users/azizulshaikh/Projects/agentic_bi_platform/documents/multimodal/charts/01_monthly_revenue_trend.png'
  file_name: '01_monthly_revenue_trend.png'
  title: 'Monthly Revenue & Order Volume — Olist Platform (Jan 2017 – Aug 2018)'
  key_insight: 'Revenue grew consistently from R$320K in Jan 2017 to a peak of R$810K in Mar 2018, followed by a gradual decline to R$660K by Aug 2018. Order volume mirrors the revenue trend, peaking at ~9,500 orders in Mar 2018.'
  chart_type: 'dual-axis line chart'
  time_period: 'January 2017 – August 2018'
  data_source: 'synthetic — modelled on Olist Brazilian E-Commerce dataset'
  needs_caption: True


## 4. Recreate `load_pdf_documents()`

A PDF creates two kinds of Documents:

```text
PDF page
├── extracted text → pdf_text
└── embedded images → pdf_image
```

The text Document is immediately embeddable.

The image Document gets `needs_caption=True` and an extracted image file path so the captioner can process it later.


In [7]:
# 7. Recreate load_pdf_documents()

def load_pdf_documents(
    pdf_dir: Path,
    metadata_lookup: dict,
    min_text_length: int = 80
) -> list[Document]:

    from pypdf import PdfReader

    docs = []

    if not pdf_dir.exists():
        logger.warning("No PDF directory found: %s", pdf_dir)
        return docs

    pdf_files = sorted(pdf_dir.glob("*.pdf"))

    for pdf_path in pdf_files:
        file_meta = metadata_lookup.get(pdf_path.name, {})

        try:
            reader = PdfReader(str(pdf_path))
        except Exception as exc:
            logger.error("Failed to read %s: %s", pdf_path.name, exc)
            continue

        for page_idx, page in enumerate(reader.pages):

            # ---- TEXT BRANCH ----
            text = (page.extract_text() or "").strip()

            if len(text) >= min_text_length:
                docs.append(
                    Document(
                        page_content=text,
                        metadata={
                            "source_type": "pdf_text",
                            "file_path": str(pdf_path.resolve()),
                            "file_name": pdf_path.name,
                            "title": file_meta.get("title", pdf_path.stem),
                            "page_number": page_idx + 1,
                            "key_topics": file_meta.get("key_topics", []),
                            "document_type": file_meta.get("document_type", "report"),
                            "time_period": file_meta.get("time_period", ""),
                            "data_source": file_meta.get("data_source", "synthetic"),
                            "needs_caption": False,
                        },
                    )
                )

            # ---- IMAGE BRANCH ----
            if hasattr(page, "images"):
                for img_idx, img_obj in enumerate(page.images):

                    img_name = f"{pdf_path.stem}_page{page_idx+1}_img{img_idx}.png"
                    img_save_path = pdf_path.parent / "_extracted" / img_name
                    img_save_path.parent.mkdir(exist_ok=True)

                    try:
                        img_save_path.write_bytes(img_obj.data)
                    except Exception as exc:
                        logger.warning("Could not save %s: %s", img_name, exc)
                        continue

                    docs.append(
                        Document(
                            page_content="",
                            metadata={
                                "source_type": "pdf_image",
                                "file_path": str(img_save_path.resolve()),
                                "file_name": img_name,
                                "parent_pdf": pdf_path.name,
                                "title": file_meta.get("title", pdf_path.stem),
                                "page_number": page_idx + 1,
                                "image_index": img_idx,
                                "document_type": file_meta.get("document_type", "report"),
                                "time_period": file_meta.get("time_period", ""),
                                "data_source": file_meta.get("data_source", "synthetic"),
                                "needs_caption": True,
                            },
                        )
                    )

    logger.info("PDF loading complete: %d Documents", len(docs))
    return docs


In [8]:
# 8. Execute PDF loading and inspect both branches

pdf_docs = load_pdf_documents(CORPUS_DIR / "pdfs", pdfs_meta)

print("PDF-derived Documents:", len(pdf_docs))
print("Counts:", Counter(d.metadata.get("source_type") for d in pdf_docs))

for source_type in ["pdf_text", "pdf_image"]:
    matches = [d for d in pdf_docs if d.metadata.get("source_type") == source_type]

    if matches:
        d = matches[0]
        print(f"\n--- {source_type} example ---")
        print("page_content chars:", len(d.page_content))
        print("metadata:", d.metadata)


INFO: PDF loading complete: 25 Documents


PDF-derived Documents: 25
Counts: Counter({'pdf_text': 13, 'pdf_image': 12})

--- pdf_text example ---
page_content chars: 324
metadata: {'source_type': 'pdf_text', 'file_path': '/Users/azizulshaikh/Projects/agentic_bi_platform/documents/multimodal/pdfs/P01_q2_2018_business_report.pdf', 'file_name': 'P01_q2_2018_business_report.pdf', 'title': 'Olist E-Commerce Platform — Q2 2018 Business Performance Report', 'page_number': 1, 'key_topics': ['Q2 2018 revenue decline of 4.7% QoQ', 'delivery time deterioration as primary cause', 'return rate increase to 5.2%', 'regional performance', 'category return rates', 'recommendations'], 'document_type': 'quarterly business report', 'time_period': 'Q2 2018 (April – June 2018)', 'data_source': 'synthetic — modelled on Olist Brazilian E-Commerce dataset', 'needs_caption': False}

--- pdf_image example ---
page_content chars: 0
metadata: {'source_type': 'pdf_image', 'file_path': '/Users/azizulshaikh/Projects/agentic_bi_platform/documents/multimodal/pd

## 5. Recreate `load_multimodal_corpus()`

Now we compose the individual loaders.

This is the first major output boundary:

```python
all_docs: list[Document]
```

All source types are flattened into one collection, while their identity is preserved in metadata.


In [9]:
# 9. Recreate load_multimodal_corpus()

def load_multimodal_corpus(multimodal_dir: Path) -> list[Document]:

    meta_dir = multimodal_dir / "metadata"
    charts_dir = multimodal_dir / "charts"
    tables_dir = multimodal_dir / "tables"
    pdfs_dir = multimodal_dir / "pdfs"

    charts_meta = load_metadata_json(meta_dir, "charts_metadata.json")
    tables_meta = load_metadata_json(meta_dir, "tables_metadata.json")
    pdfs_meta = load_metadata_json(meta_dir, "pdfs_metadata.json")

    all_docs = []

    if charts_dir.exists():
        all_docs.extend(
            load_image_documents(charts_dir, "chart", charts_meta)
        )

    if tables_dir.exists():
        all_docs.extend(
            load_image_documents(tables_dir, "table", tables_meta)
        )

    if pdfs_dir.exists():
        all_docs.extend(
            load_pdf_documents(pdfs_dir, pdfs_meta)
        )

    needs_caption = sum(
        1 for d in all_docs
        if d.metadata.get("needs_caption")
    )

    logger.info(
        "Corpus loaded: %d total, %d need captioning, %d text-ready",
        len(all_docs),
        needs_caption,
        len(all_docs) - needs_caption,
    )

    return all_docs


In [10]:
# 10. Execute the complete loader

all_docs = load_multimodal_corpus(CORPUS_DIR)

print("\n========== LOADER OUTPUT ==========")
print("Total:", len(all_docs))
print("Source types:", Counter(d.metadata.get("source_type") for d in all_docs))
print("needs_caption:", Counter(bool(d.metadata.get("needs_caption")) for d in all_docs))

for i, d in enumerate(all_docs[:5]):
    print(f"\n[{i}] {d.metadata.get('source_type')} | {d.metadata.get('file_name')}")
    print("content chars:", len(d.page_content))
    print("needs_caption:", d.metadata.get("needs_caption"))


INFO: Loaded 10 chart documents
INFO: Loaded 5 table documents
INFO: PDF loading complete: 25 Documents
INFO: Corpus loaded: 40 total, 27 need captioning, 13 text-ready



========== LOADER OUTPUT ==========
Total: 40
Source types: Counter({'pdf_text': 13, 'pdf_image': 12, 'chart': 10, 'table': 5})
needs_caption: Counter({True: 27, False: 13})

[0] chart | 01_monthly_revenue_trend.png
content chars: 213
needs_caption: True

[1] chart | 02_quarterly_revenue_bar.png
content chars: 197
needs_caption: True

[2] chart | 03_category_revenue_bar.jpg
content chars: 250
needs_caption: True

[3] chart | 04_category_return_rate.png
content chars: 275
needs_caption: True

[4] chart | 05_regional_revenue_pie.png
content chars: 293
needs_caption: True


# Part II — Rebuild `captioner.py`

## 6. Image bytes → multimodal model input

First we need the raw image represented as:

```text
Path
 ↓
bytes
 ↓
base64 string
```

plus its MIME type.

This is independent of the LLM.


In [11]:
# 11. Recreate image_to_base64()

IMAGE_MEDIA_TYPES = {
    ".png": "image/png",
    ".jpg": "image/jpeg",
    ".jpeg": "image/jpeg",
    ".webp": "image/webp",
}

def image_to_base64(image_path: Path) -> tuple[str, str]:

    suffix = image_path.suffix.lower()
    media_type = IMAGE_MEDIA_TYPES.get(suffix)

    if media_type is None:
        raise ValueError(
            f"Unsupported image format: {suffix!r}. "
            f"Supported: {list(IMAGE_MEDIA_TYPES)}"
        )

    with open(image_path, "rb") as f:
        data = base64.standard_b64encode(f.read()).decode("utf-8")

    return data, media_type


In [12]:
# 12. Execute image_to_base64() on one actual image

image_doc = next(
    (d for d in all_docs if d.metadata.get("source_type") in {"chart", "table", "pdf_image"}),
    None
)

if image_doc:
    image_path = Path(image_doc.metadata["file_path"])
    b64_data, media_type = image_to_base64(image_path)

    print("Image:", image_path.name)
    print("Media type:", media_type)
    print("Base64 type:", type(b64_data).__name__)
    print("Base64 length:", len(b64_data))
    print("Preview:", b64_data[:80] + "...")
else:
    print("No image document found.")


Image: 01_monthly_revenue_trend.png
Media type: image/png
Base64 type: str
Base64 length: 158988
Preview: iVBORw0KGgoAAAANSUhEUgAABvYAAALdCAYAAAD3U1RXAAAAOnRFWHRTb2Z0d2FyZQBNYXRwbG90bGli...


## 7. Build the caption prompt

The caption itself is the future embedding representation of the visual.

Therefore the prompt is designed around what a business user is likely to query:

- chart/table type;
- axes/columns;
- business finding;
- anomalies/highlights;
- period/scope;
- visible numbers without inventing values.


In [13]:
# 13. Recreate the caption prompt

CAPTION_PROMPT = """\
You are a business intelligence analyst describing a chart or table image for a searchable knowledge base.

Describe this image in 150–250 words covering ALL of the following:
1. Visual type (e.g. bar chart, line chart, pie chart, data table, dual-axis chart)
2. What each axis or column represents, including units
3. The primary business finding or trend the visual communicates
4. Any anomalies, highlights, or colour-coded thresholds visible
5. The time period or dataset scope shown

Write in plain prose, not bullet points. Be specific about numbers where clearly readable. Do not invent numbers that are not visible in the image. Start directly with the description — no preamble."""

print(CAPTION_PROMPT)


You are a business intelligence analyst describing a chart or table image for a searchable knowledge base.

Describe this image in 150–250 words covering ALL of the following:
1. Visual type (e.g. bar chart, line chart, pie chart, data table, dual-axis chart)
2. What each axis or column represents, including units
3. The primary business finding or trend the visual communicates
4. Any anomalies, highlights, or colour-coded thresholds visible
5. The time period or dataset scope shown

Write in plain prose, not bullet points. Be specific about numbers where clearly readable. Do not invent numbers that are not visible in the image. Start directly with the description — no preamble.


In [14]:
# 14. Build the actual LangChain multimodal message

from langchain_core.messages import HumanMessage

def build_caption_message(
    image_path: Path,
    extra_context: str | None = None
) -> HumanMessage:

    b64_data, media_type = image_to_base64(image_path)

    prompt_text = CAPTION_PROMPT

    if extra_context:
        prompt_text += (
            "\n\nAdditional context from the data source "
            "(use to verify numbers you can see):\n"
            + extra_context
        )

    return HumanMessage(
        content=[
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:{media_type};base64,{b64_data}",
                    "detail": "high",
                },
            },
            {
                "type": "text",
                "text": prompt_text,
            },
        ]
    )

if image_doc:
    message = build_caption_message(
        Path(image_doc.metadata["file_path"]),
        image_doc.metadata.get("key_insight") or None
    )

    print("Message type:", type(message).__name__)
    print("Number of content parts:", len(message.content))
    print("Part 0:", message.content[0]["type"])
    print("Part 1:", message.content[1]["type"])
    print("Image detail:", message.content[0]["image_url"]["detail"])
    print("Prompt chars:", len(message.content[1]["text"]))


Message type: HumanMessage
Number of content parts: 2
Part 0: image_url
Part 1: text
Image detail: high
Prompt chars: 979


## 8. Directly instantiate the vision model

The production helper hides this behind `get_multimodal_llm()`.

In this research notebook we intentionally expose the choice so that we can see exactly what the function is doing before turning it into a reusable helper.


In [15]:
# 15. Direct vision model construction

from langchain_openai import ChatOpenAI

vision_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
)

print(type(vision_llm).__name__)
print("model:", vision_llm.model_name)


ChatOpenAI
model: gpt-4o-mini


## 9. Recreate `caption_image()`

The function boundary is now completely visible:

```text
image_path
   ↓
build_caption_message()
   ↓
HumanMessage
   ↓
vision_llm.invoke()
   ↓
AI response
   ↓
response.content.strip()
   ↓
str
```


In [16]:
# 16. Recreate caption_image()

def caption_image(
    image_path: Path,
    extra_context: str | None = None
) -> str:

    message = build_caption_message(
        image_path,
        extra_context=extra_context
    )

    response = vision_llm.invoke([message])

    caption = response.content.strip()

    logger.info(
        "Captioned %s: %d chars",
        image_path.name,
        len(caption)
    )

    return caption


In [17]:
# 17. Controlled experiment: caption ONE real image

RUN_ONE_VISION_CALL = False

if RUN_ONE_VISION_CALL and image_doc:
    caption = caption_image(
        Path(image_doc.metadata["file_path"]),
        extra_context=image_doc.metadata.get("key_insight") or None
    )

    print("========== CAPTION ==========")
    print(caption)
    print("\nCharacters:", len(caption))
    print("Words:", len(caption.split()))
else:
    print("Set RUN_ONE_VISION_CALL = True for one real vision call.")


Set RUN_ONE_VISION_CALL = True for one real vision call.


## 10. Recreate `caption_image_safe()`

The safe wrapper is important for batch processing.

If one image fails:

```text
vision failure
     ↓
fallback exists?
   /       \
 yes       no
  ↓         ↓
return    raise
fallback
```

For this pipeline, the `key_insight` can serve as the fallback.


In [18]:
# 18. Recreate caption_image_safe()

def caption_image_safe(
    image_path: Path,
    extra_context: str | None = None,
    fallback: str | None = None,
) -> str:

    try:
        return caption_image(
            image_path,
            extra_context=extra_context
        )

    except Exception as exc:
        logger.warning(
            "Failed to caption %s: %s — %s",
            image_path.name,
            type(exc).__name__,
            exc
        )

        if fallback is not None:
            return fallback

        raise


# Part III — Rebuild `ingest.py`

## 11. Decide which Documents require vision

This is the first orchestration decision.

A Document should be captioned when:

1. `needs_caption=True`;
2. the image is not below the 5 KB threshold.


In [19]:
# 19. Recreate _should_caption()

MIN_IMAGE_BYTES = 5_000
MAX_CAPTION_WORKERS = 4

def should_caption(doc: Document) -> bool:

    if not doc.metadata.get("needs_caption", False):
        return False

    file_path = doc.metadata.get("file_path", "")

    if file_path and Path(file_path).exists():
        size = Path(file_path).stat().st_size

        if size < MIN_IMAGE_BYTES:
            logger.info(
                "Skipping tiny image (%d bytes): %s",
                size,
                doc.metadata.get("file_name")
            )
            return False

    return True

to_caption = [d for d in all_docs if should_caption(d)]
text_ready = [d for d in all_docs if not should_caption(d)]

print("Total:", len(all_docs))
print("To caption:", len(to_caption))
print("Text ready:", len(text_ready))
print("To caption by type:", Counter(d.metadata.get("source_type") for d in to_caption))
print("Text ready by type:", Counter(d.metadata.get("source_type") for d in text_ready))


Total: 40
To caption: 27
Text ready: 13
To caption by type: Counter({'pdf_image': 12, 'chart': 10, 'table': 5})
Text ready by type: Counter({'pdf_text': 13})


## 12. Recreate the caption cache

Cache key:

```text
file_path::mtime
```

This means a changed file automatically gets a new caption.

`force_recaption=True` is implemented by starting with an empty cache.


In [20]:
# 20. Recreate cache functions

def cache_path(index_dir: Path) -> Path:
    return index_dir / "caption_cache.json"

def load_cache(index_dir: Path) -> dict:
    path = cache_path(index_dir)

    if path.exists():
        with open(path, "r") as f:
            return json.load(f)

    return {}

def save_cache(index_dir: Path, cache: dict) -> None:
    index_dir.mkdir(parents=True, exist_ok=True)

    with open(cache_path(index_dir), "w") as f:
        json.dump(cache, f, indent=2)

def cache_key(file_path: str) -> str:
    try:
        mtime = os.path.getmtime(file_path)
        return f"{file_path}::{mtime:.0f}"
    except FileNotFoundError:
        return file_path

cache = load_cache(INDEX_DIR)

print("Cache path:", cache_path(INDEX_DIR))
print("Existing entries:", len(cache))

if to_caption:
    print("Example key:", cache_key(to_caption[0].metadata["file_path"]))


Cache path: /Users/azizulshaikh/Projects/agentic_bi_platform/data/vectorstore/faiss_multimodal/caption_cache.json
Existing entries: 27
Example key: /Users/azizulshaikh/Projects/agentic_bi_platform/documents/multimodal/charts/01_monthly_revenue_trend.png::1789576814


## 13. Recreate `_caption_one()`

This is the atomic unit of the batch process.

Input:

```python
Document + cache
```

Output:

```python
(Document, was_cached)
```

The critical mutation is:

```python
doc.page_content = caption
```


In [21]:
# 21. Recreate _caption_one()

def caption_one(
    doc: Document,
    cache: dict,
    index_dir: Path
) -> tuple[Document, bool]:

    file_path = doc.metadata.get("file_path", "")
    key = cache_key(file_path)

    if key in cache:
        doc.page_content = cache[key]
        return doc, True

    extra_context = doc.metadata.get("key_insight") or None

    caption = caption_image_safe(
        Path(file_path),
        extra_context=extra_context,
        fallback=doc.metadata.get("key_insight", ""),
    )

    doc.page_content = caption
    cache[key] = caption

    return doc, False


In [22]:
# 22. Unit-test the cache path without an API call

if to_caption:
    test_doc = to_caption[0]
    test_cache = {
        cache_key(test_doc.metadata["file_path"]):
        "TEST CACHED CAPTION"
    }

    updated_doc, was_cached = caption_one(
        test_doc,
        test_cache,
        INDEX_DIR
    )

    print("was_cached:", was_cached)
    print("page_content:", updated_doc.page_content)
    print("file_name:", updated_doc.metadata["file_name"])
else:
    print("No image document available.")


was_cached: True
page_content: TEST CACHED CAPTION
file_name: 01_monthly_revenue_trend.png


## 14. Recreate `_run_captioning()`

Now we move from one image to many.

The important behavior is:

```text
submit N independent caption jobs
          ↓
ThreadPoolExecutor
          ↓
as_completed()
          ↓
put result back at original index
```

Therefore network completion order does not determine document order.


In [23]:
# 23. Recreate _run_captioning()

def run_captioning(
    docs_needing_caption: list[Document],
    cache: dict,
    index_dir: Path,
    max_workers: int
) -> list[Document]:

    if not docs_needing_caption:
        return []

    results = [None] * len(docs_needing_caption)
    cache_hits = 0
    api_calls = 0

    with ThreadPoolExecutor(max_workers=max_workers) as pool:

        future_to_idx = {
            pool.submit(
                caption_one,
                doc,
                cache,
                index_dir
            ): idx
            for idx, doc in enumerate(docs_needing_caption)
        }

        for future in as_completed(future_to_idx):

            idx = future_to_idx[future]

            try:
                doc, was_cached = future.result()

                results[idx] = doc

                if was_cached:
                    cache_hits += 1
                else:
                    api_calls += 1

            except Exception as exc:
                logger.error(
                    "Captioning failed for doc %d (%s): %s",
                    idx,
                    docs_needing_caption[idx].metadata.get("file_name", "?"),
                    exc
                )

                results[idx] = docs_needing_caption[idx]

    save_cache(index_dir, cache)

    logger.info(
        "Captioning complete: %d cache hits, %d new API calls",
        cache_hits,
        api_calls
    )

    return results


In [24]:
# 24. Execute the real batch captioning stage when ready

RUN_BATCH_CAPTIONING = False

if RUN_BATCH_CAPTIONING:
    cache = load_cache(INDEX_DIR)

    captioned_docs = run_captioning(
        docs_needing_caption=to_caption,
        cache=cache,
        index_dir=INDEX_DIR,
        max_workers=MAX_CAPTION_WORKERS
    )

    print("Captioned:", len(captioned_docs))

    for d in captioned_docs[:3]:
        print("\n---", d.metadata["file_name"], "---")
        print(d.page_content[:500])
else:
    captioned_docs = []
    print("Set RUN_BATCH_CAPTIONING = True to make the real vision calls.")


Set RUN_BATCH_CAPTIONING = True to make the real vision calls.


# Part IV — Prepare documents for embedding

## 15. Merge + filter

After captioning:

```python
all_ready = text_ready + captioned
```

Then only documents with at least 20 non-whitespace characters are allowed into the embedding stage.

This is the exact point where we transition from **multimodal ingestion logic** to **normal text retrieval infrastructure**.


In [25]:
# 25. Recreate the final content filter

def filter_embeddable_documents(
    docs: list[Document],
    min_chars: int = 20
) -> list[Document]:

    embeddable = [
        d for d in docs
        if d.page_content
        and len(d.page_content.strip()) >= min_chars
    ]

    skipped = len(docs) - len(embeddable)

    if skipped:
        logger.warning(
            "Skipped %d Documents with empty/short content",
            skipped
        )

    return embeddable

if RUN_BATCH_CAPTIONING:
    all_ready = text_ready + captioned_docs
else:
    # Research-only mode: keep placeholders so we can inspect the filter.
    all_ready = text_ready + to_caption

embeddable = filter_embeddable_documents(all_ready)

print("all_ready:", len(all_ready))
print("embeddable:", len(embeddable))
print("skipped:", len(all_ready) - len(embeddable))
print("By source:", Counter(d.metadata.get("source_type") for d in embeddable))


all_ready: 40
embeddable: 27
skipped: 13
By source: Counter({'pdf_text': 13, 'chart': 9, 'table': 5})


In [26]:
# 26. Inspect the exact data entering the embedding stage

for i, d in enumerate(embeddable[:5]):
    print(f"\n===== Document {i} =====")
    print("source_type:", d.metadata.get("source_type"))
    print("file_name:", d.metadata.get("file_name"))
    print("page_number:", d.metadata.get("page_number"))
    print("parent_pdf:", d.metadata.get("parent_pdf"))
    print("page_content type:", type(d.page_content).__name__)
    print("page_content chars:", len(d.page_content))
    print("preview:", d.page_content[:300].replace("\n", " "))



===== Document 0 =====
source_type: pdf_text
file_name: P01_q2_2018_business_report.pdf
page_number: 1
parent_pdf: None
page_content type: str
page_content chars: 324
preview: Olist E-Commerce Platform  Q2 2018 Business Performance Report  April – June 2018 | Confidential | Prepared by: BI Analytics Team  R$2,250,000  26,300  R$85.55  5.2% Total Revenue  Total Orders  Avg Order Value  Return Rate  24,800  22.4%  4.12 / 5.00  10.6 days Unique Customers  Repeat Rate  Avg Re

===== Document 1 =====
source_type: pdf_text
file_name: P01_q2_2018_business_report.pdf
page_number: 2
parent_pdf: None
page_content type: str
page_content chars: 1416
preview: 1. Executive Summary Q2 2018 marks the first quarter-over-quarter revenue decline since platform launch, with total GMV falling 4.7% to R$2,250,000 compared to R$2,360,000 in Q1 2018. Order volume contracted 5.4% to 26,300 orders. Despite these top-line pressures, year-over-year performance remains 

===== Document 2 =====
source_type: pdf_te

# Part V — Embeddings and FAISS

## 16. Make the embedding model explicit

The production ingestion code obtains embeddings through `get_embeddings()`.

Here we expose the research choice directly:

```text
text-embedding-3-small
```

This is why the captioning strategy works: once the image is converted to text, the same text embedding pipeline can be used.


In [27]:
# 27. Instantiate the embedding model directly

from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print(type(embedding_model).__name__)
print("model:", embedding_model.model)


OpenAIEmbeddings
model: text-embedding-3-small


In [28]:
# 28. Inspect one embedding output

RUN_ONE_EMBEDDING = False

if RUN_ONE_EMBEDDING and embeddable:
    sample_text = embeddable[0].page_content
    vector = embedding_model.embed_query(sample_text)

    print("Input type:", type(sample_text).__name__)
    print("Input chars:", len(sample_text))
    print("Output type:", type(vector).__name__)
    print("Vector length:", len(vector))
    print("First 10 values:", vector[:10])
else:
    print("Set RUN_ONE_EMBEDDING = True for one real embedding call.")


Set RUN_ONE_EMBEDDING = True for one real embedding call.


## 17. Recreate `_build_and_save()`

FAISS receives the final `Document` list and the embedding model.

Conceptually:

```text
Document.page_content
        ↓
embedding model
        ↓
vector
        ↓
FAISS
```

Metadata stays associated with the Document so retrieval can return provenance.


In [29]:
# 29. Recreate _build_and_save()

from langchain_community.vectorstores import FAISS

def build_and_save(
    docs: list[Document],
    index_dir: Path
) -> FAISS:

    if not docs:
        raise ValueError(
            "No documents to index — corpus appears empty."
        )

    vectorstore = FAISS.from_documents(
        docs,
        embedding_model
    )

    index_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    vectorstore.save_local(
        str(index_dir)
    )

    logger.info(
        "FAISS index saved to %s",
        index_dir
    )

    return vectorstore


/var/folders/1w/k2xkpg3s50n1nnp_jrdp_pdw0000gn/T/ipykernel_73930/604442647.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [30]:
# 30. Build the actual FAISS index

RUN_BUILD_FAISS = False

if RUN_BUILD_FAISS:
    vectorstore = build_and_save(
        embeddable,
        INDEX_DIR
    )

    print("Vectorstore:", type(vectorstore).__name__)
    print("Index exists:", INDEX_DIR.exists())

    for p in sorted(INDEX_DIR.iterdir()):
        print(" -", p.name)
else:
    vectorstore = None
    print("Set RUN_BUILD_FAISS = True to build the real index.")


Set RUN_BUILD_FAISS = True to build the real index.


# Part VI — Reload + retrieve

## 18. Recreate `load_multimodal_vectorstore()`

The persisted FAISS index must be reloadable using the same embedding model configuration.


In [31]:
# 31. Recreate load_multimodal_vectorstore()

def load_multimodal_vectorstore(index_dir: Path) -> FAISS:

    if not index_dir.exists():
        raise FileNotFoundError(
            f"Multimodal index not found at {index_dir}. "
            "Build the index first."
        )

    return FAISS.load_local(
        str(index_dir),
        embedding_model,
        allow_dangerous_deserialization=True
    )


In [32]:
# 32. Reload the persisted index

RUN_RELOAD = False

if RUN_RELOAD:
    loaded_vectorstore = load_multimodal_vectorstore(INDEX_DIR)
    print("Loaded:", type(loaded_vectorstore).__name__)
else:
    loaded_vectorstore = None
    print("Set RUN_RELOAD = True after building the index.")


Set RUN_RELOAD = True after building the index.


## 19. Retrieval smoke test

The retrieval contract is:

```python
similarity_search(query, k=3)
        ↓
list[Document]
```

We inspect both:

```python
doc.page_content
doc.metadata
```

The metadata is what lets the downstream RAG answer identify the source chart/table/PDF/page.


In [33]:
# 33. Execute retrieval

QUERY = "return rate by product category"

if loaded_vectorstore is not None:

    results = loaded_vectorstore.similarity_search(
        QUERY,
        k=3
    )

    print("Query:", QUERY)
    print("Results:", len(results))

    for i, doc in enumerate(results, 1):
        print(f"\n===== RESULT {i} =====")
        print("source_type:", doc.metadata.get("source_type"))
        print("file_name:", doc.metadata.get("file_name"))
        print("title:", doc.metadata.get("title"))
        print("page_number:", doc.metadata.get("page_number"))
        print("parent_pdf:", doc.metadata.get("parent_pdf"))
        print("content:")
        print(doc.page_content[:800])

else:
    print("Build and reload the FAISS index first.")


Build and reload the FAISS index first.


# Part VII — Recreate the complete `ingest_multimodal_corpus()`

## 20. Only now do we compose the functions

This mirrors the production orchestration, but every component below was built and inspected in previous cells.

```text
load
 ↓
route
 ↓
caption
 ↓
merge
 ↓
filter
 ↓
embed
 ↓
save
```


In [34]:
# 34. Recreate ingest_multimodal_corpus()

def ingest_multimodal_corpus(
    multimodal_dir: Path | None = None,
    max_caption_workers: int = MAX_CAPTION_WORKERS,
    force_recaption: bool = False,
) -> FAISS:

    if multimodal_dir is None:
        multimodal_dir = CORPUS_DIR

    index_dir = INDEX_DIR

    logger.info("=== Multimodal RAG Ingestion ===")
    logger.info("Corpus dir: %s", multimodal_dir)
    logger.info("Index dir : %s", index_dir)

    # 1. Load
    all_docs = load_multimodal_corpus(
        multimodal_dir
    )

    if not all_docs:
        raise FileNotFoundError(
            f"No documents found in {multimodal_dir}"
        )

    # 2. Route
    to_caption = [
        d for d in all_docs
        if should_caption(d)
    ]

    text_ready = [
        d for d in all_docs
        if not should_caption(d)
    ]

    # 3. Caption
    cache = (
        {}
        if force_recaption
        else load_cache(index_dir)
    )

    captioned = run_captioning(
        to_caption,
        cache,
        index_dir,
        max_caption_workers
    )

    # 4. Merge
    all_ready = text_ready + captioned

    # 5. Filter
    embeddable = filter_embeddable_documents(
        all_ready
    )

    # 6. Embed + persist
    vectorstore = build_and_save(
        embeddable,
        index_dir
    )

    logger.info(
        "=== Ingestion complete: %d Documents indexed ===",
        len(embeddable)
    )

    return vectorstore


## 21. Full pipeline execution

This cell is the notebook equivalent of:

```bash
python scripts/ingest_multimodal.py
```

It calls the **research functions written in this notebook**, not the production module.

Because this can make multiple vision and embedding API calls, it is explicitly controlled by a switch.


In [35]:
# 35. Execute the complete research pipeline

RUN_FULL_PIPELINE = False

if RUN_FULL_PIPELINE:
    research_vectorstore = ingest_multimodal_corpus(
        multimodal_dir=CORPUS_DIR,
        max_caption_workers=4,
        force_recaption=False
    )

    print(
        "Pipeline complete:",
        type(research_vectorstore).__name__
    )
else:
    research_vectorstore = None
    print("Set RUN_FULL_PIPELINE = True to execute all stages.")


Set RUN_FULL_PIPELINE = True to execute all stages.


# Part VIII — Rebuild the CLI behavior

## 22. `ingest_multimodal.py` is an operational wrapper

The fourth file primarily does:

```text
CLI arguments
    ↓
logging
    ↓
ingest_multimodal_corpus()
    ↓
smoke-test retrieval
```

Its `--dry-run` mode is especially useful because it calls the loader without calling the vision API.


In [36]:
# 36. Recreate the CLI argument parser

import argparse

def build_argument_parser():
    parser = argparse.ArgumentParser(
        description="Ingest multimodal corpus into FAISS"
    )

    parser.add_argument(
        "--corpus-dir",
        type=Path,
        default=None
    )

    parser.add_argument(
        "--workers",
        type=int,
        default=4
    )

    parser.add_argument(
        "--force-recaption",
        action="store_true"
    )

    parser.add_argument(
        "--dry-run",
        action="store_true"
    )

    parser.add_argument(
        "--log-level",
        default="INFO",
        choices=["DEBUG", "INFO", "WARNING", "ERROR"]
    )

    return parser

parser = build_argument_parser()
print(parser.format_help())


usage: ipykernel_launcher.py [-h] [--corpus-dir CORPUS_DIR]
                             [--workers WORKERS] [--force-recaption]
                             [--dry-run]
                             [--log-level {DEBUG,INFO,WARNING,ERROR}]

Ingest multimodal corpus into FAISS

options:
  -h, --help            show this help message and exit
  --corpus-dir CORPUS_DIR
  --workers WORKERS
  --force-recaption
  --dry-run
  --log-level {DEBUG,INFO,WARNING,ERROR}



In [37]:
# 37. Recreate the CLI dry-run function

def dry_run(corpus_dir: Path | None = None) -> None:

    if corpus_dir is None:
        corpus_dir = CORPUS_DIR

    print("Corpus dir:", corpus_dir)

    docs = load_multimodal_corpus(corpus_dir)

    by_type = {}

    for d in docs:
        source_type = d.metadata.get(
            "source_type",
            "unknown"
        )

        by_type.setdefault(
            source_type,
            []
        ).append(d)

    print("\nTotal documents:", len(docs))

    for source_type, source_docs in sorted(by_type.items()):

        needs = sum(
            1
            for d in source_docs
            if d.metadata.get("needs_caption")
        )

        print(
            f"  {source_type:<15}"
            f"{len(source_docs):>3} documents"
            f" ({needs} need captioning)"
        )

    print("\nFiles that would be captioned:")

    for d in docs:
        if d.metadata.get("needs_caption"):
            print(
                " ",
                d.metadata.get("file_name", "?")
            )


In [38]:
# 38. Execute the dry run

dry_run(CORPUS_DIR)


INFO: Loaded 10 chart documents
INFO: Loaded 5 table documents


Corpus dir: /Users/azizulshaikh/Projects/agentic_bi_platform/documents/multimodal


INFO: PDF loading complete: 25 Documents
INFO: Corpus loaded: 40 total, 27 need captioning, 13 text-ready



Total documents: 40
  chart           10 documents (10 need captioning)
  pdf_image       12 documents (12 need captioning)
  pdf_text        13 documents (0 need captioning)
  table            5 documents (5 need captioning)

Files that would be captioned:
  01_monthly_revenue_trend.png
  02_quarterly_revenue_bar.png
  03_category_revenue_bar.jpg
  04_category_return_rate.png
  05_regional_revenue_pie.png
  06_regional_delivery_days.png
  07_payment_method_donut.jpg
  08_review_score_distribution.png
  09_customer_segment_revenue.png
  10_aov_vs_delivery.png
  T01_kpi_summary_table.png
  T02_category_performance_table.png
  T03_regional_performance_table.png
  T04_top_sellers_table.png
  T05_quarterly_comparison_table.png
  P01_q2_2018_business_report_page2_img0.png
  P01_q2_2018_business_report_page2_img1.png
  P01_q2_2018_business_report_page3_img0.png
  P01_q2_2018_business_report_page4_img0.png
  P01_q2_2018_business_report_page4_img1.png
  P01_q2_2018_business_report_page5_img0.

# Part IX — Debugging the pipeline through intermediate state

## 23. A useful research/debugging inspection

When this pipeline produces an unexpected retrieval result, inspect the data at these boundaries:

```text
1. raw file
2. loaded Document
3. caption
4. final page_content
5. embedding
6. FAISS retrieval
```

Do not jump directly to FAISS debugging. Most multimodal retrieval problems can originate much earlier — especially in PDF extraction, image filtering, metadata lookup, or caption quality.


In [39]:
# 39. Build a compact corpus audit table

audit_rows = []

for d in all_docs:
    audit_rows.append({
        "source_type": d.metadata.get("source_type"),
        "file_name": d.metadata.get("file_name"),
        "needs_caption": d.metadata.get("needs_caption"),
        "page_number": d.metadata.get("page_number"),
        "image_index": d.metadata.get("image_index"),
        "content_chars": len(d.page_content or ""),
        "has_key_insight": bool(d.metadata.get("key_insight")),
        "file_exists": Path(d.metadata.get("file_path", "")).exists(),
    })

import pandas as pd

audit_df = pd.DataFrame(audit_rows)

display(audit_df.head(20))
print("\nShape:", audit_df.shape)
print("\nSource counts:")
display(audit_df["source_type"].value_counts())


,source_type,file_name,needs_caption,page_number,image_index,content_chars,has_key_insight,file_exists
0,chart,01_monthly_revenue_trend.png,True,NaN,NaN,19,True,True
1,chart,02_quarterly_revenue_bar.png,True,NaN,NaN,197,True,True
2,chart,03_category_revenue_bar.jpg,True,NaN,NaN,250,True,True
3,chart,04_category_return_rate.png,True,NaN,NaN,275,True,True
4,chart,05_regional_revenue_pie.png,True,NaN,NaN,293,True,True
5,chart,06_regional_delivery_days.png,True,NaN,NaN,315,True,True
6,chart,07_payment_method_donut.jpg,True,NaN,NaN,355,True,True
7,chart,08_review_score_distribution.png,True,NaN,NaN,310,True,True
8,chart,09_customer_segment_revenue.png,True,NaN,NaN,372,True,True
9,chart,10_aov_vs_delivery.png,True,NaN,NaN,398,True,True



Shape: (40, 8)

Source counts:


source_type
pdf_text     13
pdf_image    12
chart        10
table         5
Name: count, dtype: int64

# 24. Final mental model

The entire implementation can now be understood as four layers:

### Layer 1 — Loading

```python
files → list[Document]
```

### Layer 2 — Visual understanding

```python
image → caption string
```

### Layer 3 — Ingestion

```python
Documents → filtered Documents → embeddings → FAISS
```

### Layer 4 — Retrieval

```python
query → similarity_search() → list[Document]
```

The central insight is that the system is **multimodal at ingestion time but text-native at retrieval time**.

That is why the final RAG layer can remain largely unchanged.


# 25. Production-module mapping

| Research notebook | Final production location |
|---|---|
| `load_metadata_json()` | `loaders.py` |
| `load_image_documents()` | `loaders.py` |
| `load_pdf_documents()` | `loaders.py` |
| `load_multimodal_corpus()` | `loaders.py` |
| `image_to_base64()` | `captioner.py` |
| `build_caption_message()` | `captioner.py` |
| direct `ChatOpenAI` construction | hidden behind `get_multimodal_llm()` |
| `caption_image()` | `captioner.py` |
| `caption_image_safe()` | `captioner.py` |
| `should_caption()` | `ingest.py` |
| cache functions | `ingest.py` |
| `caption_one()` | `ingest.py` |
| `run_captioning()` | `ingest.py` |
| `filter_embeddable_documents()` | `ingest.py` |
| `build_and_save()` | `ingest.py` |
| `load_multimodal_vectorstore()` | `ingest.py` |
| `ingest_multimodal_corpus()` | `ingest.py` |
| argument parser | `ingest_multimodal.py` |
| `dry_run()` | `ingest_multimodal.py` |

So this notebook is intentionally the **prototype from which the four production modules can be understood/reconstructed**.


# 26. Final execution checklist

For a fresh end-to-end run:

```text
Run cells 1–14
    ↓
verify Document loading
    ↓
Run ONE vision call
    ↓
inspect caption
    ↓
Run batch captioning
    ↓
inspect captioned Documents
    ↓
run filtering
    ↓
run ONE embedding
    ↓
build FAISS
    ↓
reload FAISS
    ↓
run similarity search
```

Only after these boundaries look correct should the logic be treated as production-ready.

The notebook is therefore not just a tutorial: it is a **research/debugging harness for the multimodal ingestion architecture**.
